# ACE Framework Demo

This notebook demonstrates the **Agentic Context Engineering (ACE)** framework from the paper:

> "Agentic Context Engineering: Evolving Contexts for Self-Improving Language Models" (Zhang et al., 2025)

## Overview

ACE treats contexts as **evolving playbooks** that accumulate and refine strategies through:
- **Generator**: Produces reasoning trajectories
- **Reflector**: Extracts insights from successes/failures
- **Curator**: Integrates insights via structured delta updates

In [1]:
import sys
sys.path.append('..')

from src.ace import ACEFramework
from src.ace.playbook import Playbook
from dotenv import load_dotenv

# Load API keys
load_dotenv()

print("ACE Framework loaded successfully!")

ACE Framework loaded successfully!


## 1. Initialize ACE Framework

In [2]:
# Initialize ACE with GPT-4 (you can use other models)
ace = ACEFramework(
    generator_model="gemma3",
    reflector_model="gemma3",
    curator_model="gemma3"
)

print("ACE Framework initialized!")
print(f"Initial playbook stats: {ace.playbook.get_stats()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ACE Framework initialized!
Initial playbook stats: {'total_bullets': 0, 'section_counts': {'strategies_and_hard_rules': 0, 'apis_to_use_for_specific_information': 0, 'common_mistakes_to_avoid': 0, 'verification_checklist': 0, 'formulas_and_calculations': 0, 'domain_concepts': 0}, 'helpful_feedback': 0, 'harmful_feedback': 0, 'avg_bullets_per_section': 0.0}


## 2. Example Task: Multi-step Math Problem

In [3]:
# Define a simple task
task = "If a train travels 120 km in 2 hours, then how far will it travel in 5 hours at the same speed?"
ground_truth = "300 km"

print(f"Task: {task}")
print(f"Ground Truth: {ground_truth}")

Task: If a train travels 120 km in 2 hours, then how far will it travel in 5 hours at the same speed?
Ground Truth: 300 km


## 3. Generate Initial Response (No Playbook)

In [4]:
# Generate without playbook
result_no_playbook = ace.generator.generate(
    task=task,
    playbook=None,
    task_type="qa"
)

print("=== Response Without Playbook ===")
print(f"Reasoning: {result_no_playbook['reasoning']}")
print(f"\nAnswer: {result_no_playbook['answer']}")

=== Response Without Playbook ===
Reasoning: ```json
"reasoning": "First, we need to calculate the train's speed. Speed is calculated by dividing the distance traveled by the time taken. In this case, the train travels 120 km in 2 hours. So, the speed is 120 km / 2 hours = 60 km/hour.  Now that we know the speed, we can calculate the distance traveled in 5 hours. Distance = Speed * Time.  Therefore, the distance is 60 km/hour * 5 hours = 300 km.",
"answer": "300 km"
```

Answer: ```json
{
  "reasoning": "First, we need to calculate the train's speed. Speed is calculated by dividing the distance traveled by the time taken. In this case, the train travels 120 km in 2 hours. So, the speed is 120 km / 2 hours = 60 km/hour.  Now that we know the speed, we can calculate the distance traveled in 5 hours. Distance = Speed * Time.  Therefore, the distance is 60 km/hour * 5 hours = 300 km.",
  "answer": "300 km"
}
```


## 4. Reflect on the Response

In [5]:
# Reflect on the generation
reflection = ace.reflector.reflect(
    trajectory=result_no_playbook,
    ground_truth=ground_truth,
    refine=True
)

print("=== Reflection ===")
print(f"Error Analysis: {reflection.get('error_identification', 'N/A')}")
print(f"\nRoot Cause: {reflection.get('root_cause_analysis', 'N/A')}")
print(f"\nKey Insights:")
for insight in reflection.get('key_insights', []):
    print(f"  - {insight}")

=== Reflection ===
Error Analysis: 

Root Cause: 

Key Insights:
  - ```json
{
  "reasoning": "The student demonstrates a solid ability to apply kinematic formulas but lacks a fundamental understanding of the underlying physics principles and a consistent problem-solving strategy. The primary issue is a reliance on formula application without connecting it to the concepts of motion, velocity, and time. This manifests as a lack of justification for chosen equations and an inability to translate word problems into a coherent physics model. A deeper root cause appears to be a deficit in developing a flexible, intuitive understanding of how these concepts interact, hindering their ability to choose the correct formula and interpret results meaningfully. The student needs to move beyond rote calculation and cultivate a more holistic grasp of physics principles.",
  "error_identification": "The solution is mathematically correct but lacks a clear explanation of the physics principles applied

## 5. Curate Insights into Playbook

In [6]:
# Curate insights
delta_bullets = ace.curator.curate(
    insights=[reflection],
    current_playbook=ace.playbook,
    task_context=task
)

print(f"=== Curated {len(delta_bullets)} New Bullets ===")
for bullet in delta_bullets:
    print(f"\nSection: {bullet['section']}")
    print(f"Content: {bullet['content']}")

# Update playbook
ace.playbook.update(delta_bullets)
print(f"\nUpdated playbook stats: {ace.playbook.get_stats()}")

=== Curated 0 New Bullets ===

Updated playbook stats: {'total_bullets': 0, 'section_counts': {'strategies_and_hard_rules': 0, 'apis_to_use_for_specific_information': 0, 'common_mistakes_to_avoid': 0, 'verification_checklist': 0, 'formulas_and_calculations': 0, 'domain_concepts': 0}, 'helpful_feedback': 0, 'harmful_feedback': 0, 'avg_bullets_per_section': 0.0}


## 6. Generate With Evolved Playbook

In [ ]:
# Generate with playbook
result_with_playbook = ace.generator.generate(
    task=task,
    playbook=ace.playbook,
    task_type="qa"
)

print("=== Response With Evolved Playbook ===")
print(f"Reasoning: {result_with_playbook['reasoning']}")
print(f"\nAnswer: {result_with_playbook['answer']}")
print(f"\nUsed Bullets: {result_with_playbook.get('used_bullets', [])}")

## 7. View Playbook Content

In [7]:
# Display playbook as it would appear in prompts
playbook_text = ace.playbook.to_prompt(query=task)
print("=== Evolved Playbook ===")
print(playbook_text)

=== Evolved Playbook ===
PLAYBOOK_BEGIN

PLAYBOOK_END


## 8. Offline Adaptation on Multiple Examples

In [9]:
# Define training examples
train_data = [
    {
        "task": "A car uses 8 liters of fuel to travel 100 km. How much fuel does it need for 450 km?",
        "answer": "36 liters",
        "task_type": "qa"
    },
    {
        "task": "If 5 workers can complete a job in 12 days, how many days will 3 workers take?",
        "answer": "20 days",
        "task_type": "qa"
    },
    {
        "task": "A recipe calls for 3 eggs to make 12 cookies. How many eggs are needed for 20 cookies?",
        "answer": "5 eggs",
        "task_type": "qa"
    }
]

# Run offline adaptation
evolved_playbook = ace.offline_adaptation(
    train_data=train_data,
    num_epochs=2,
    batch_size=1
)

print("\n=== Offline Adaptation Complete ===")
print(f"Final playbook stats: {evolved_playbook.get_stats()}")

Starting offline adaptation: 3 samples, 2 epochs

=== Epoch 1/2 ===


Epoch 1:  33%|███▎      | 1/3 [06:56<13:52, 416.01s/it]


KeyboardInterrupt: 

## 9. Test on New Examples

In [ ]:
# Test examples
test_data = [
    {
        "task": "A factory produces 240 units in 8 hours. How many units will it produce in 15 hours?",
        "answer": "450 units",
        "task_type": "qa"
    },
    {
        "task": "If 7 meters of fabric costs $35, how much will 12 meters cost?",
        "answer": "$60",
        "task_type": "qa"
    }
]

# Evaluate
results = ace.evaluate(test_data, use_playbook=True)

print("=== Test Results ===")
print(f"Accuracy: {results['accuracy']:.2%}")
print(f"Correct: {results['correct']}/{results['total']}")

## 10. Save and Load Playbook

In [ ]:
# Save playbook
ace.save_playbook("../results/demo_playbook.json")
print("Playbook saved to: results/demo_playbook.json")

# Load playbook
ace_new = ACEFramework()
ace_new.load_playbook("../results/demo_playbook.json")
print(f"Loaded playbook stats: {ace_new.playbook.get_stats()}")

## Summary

This notebook demonstrated:

1. **Initializing** the ACE framework
2. **Generating** responses with and without context
3. **Reflecting** on outputs to extract insights
4. **Curating** insights into playbook bullets
5. **Evolving** context through offline adaptation
6. **Evaluating** on test data

The key insight: **contexts should be evolving playbooks**, not static prompts!

### Next Steps

- Try larger datasets (see `data/README.md`)
- Run full experiments with `experiments/run_agent_experiments.py`
- Experiment with different models and settings
- Analyze playbook evolution over time